# Lemma Analysis: Handling Projections and Accessors

## Problem

Some lemmas in tactics are actually **projections or accessors** of larger lemmas, and we're breaking them down incorrectly.

### Example

```
rw [norm_div, mem_sphere_zero_iff_norm.1 x.coe_prop, mem_sphere_zero_iff_norm.1 y.coe_prop, div_one]
```

In this case:
- `mem_sphere_zero_iff_norm.1` is a projection/accessor (the `.1` part)
- The full lemma name is `mem_sphere_zero_iff_norm`
- We should check if the **full name** (including the projection) matches first
- Only if the full name doesn't match, then we should break it down

## Current Behavior

Currently, we might be:
1. Extracting `mem_sphere_zero_iff_norm.1` as a candidate
2. Breaking it down to `mem_sphere_zero_iff_norm` and `1`
3. Trying to match `mem_sphere_zero_iff_norm` (which is correct)
4. But we might miss cases where `mem_sphere_zero_iff_norm.1` itself exists as a full name

## Proposed Solution

1. **First, try to match the full candidate name** (e.g., `mem_sphere_zero_iff_norm.1`)
   - Check if this exact name exists in corpus
   - If found, use it

2. **If not found, then break it down**
   - Split on `.` to get parts
   - Try matching the base name (e.g., `mem_sphere_zero_iff_norm`)
   - Handle projections/accessors appropriately

3. **Consider the context**
   - Projections like `.1`, `.2` are typically tuple/record accessors
   - These should be resolved to the parent lemma name
   - But we should verify the parent exists first

## Implementation Notes

- Update `normalize_candidate()` to check full name first
- Update `resolve_candidate()` to handle projection patterns
- Consider Lean naming conventions:
  - `.1`, `.2`, etc. = tuple/record projections
  - `.mp`, `.mpr` = modus ponens variants
  - Other suffixes might be specific accessors

## References

- Example from tactics: `mem_sphere_zero_iff_norm.1 x.coe_prop`
- Need to check if `mem_sphere_zero_iff_norm.1` exists as a full name
- If not, resolve to `mem_sphere_zero_iff_norm` and verify it exists

# getting lemmas


In [19]:
import importlib.util
import sys

# Load and register myutils_reprover
spec_reprover = importlib.util.spec_from_file_location("myutils_reprover", "00_myutils_reprover.py")
module_reprover = importlib.util.module_from_spec(spec_reprover)
sys.modules["myutils_reprover"] = module_reprover
spec_reprover.loader.exec_module(module_reprover)
from myutils_reprover import *

# Load and register myutils2
spec = importlib.util.spec_from_file_location("myutils2", "00_myutils2.py")
module = importlib.util.module_from_spec(spec)
sys.modules["myutils2"] = module
spec.loader.exec_module(module)
from myutils2 import *


# Reload myutils2 to get updated extract_all_premises with version parameter
import importlib

# Force reload
if 'myutils2' in sys.modules:
    del sys.modules['myutils2']

import importlib.util
spec = importlib.util.spec_from_file_location("myutils2", "00_myutils2.py")
module = importlib.util.module_from_spec(spec)
sys.modules["myutils2"] = module
spec.loader.exec_module(module)

from myutils2 import extract_all_premises, load_corpus_premises

In [9]:
import json
import json
data = json.load(open("complete_proofs.json", "rt"))


premise_registry = load_corpus_premises("corpus.jsonl")
print(f"Loaded {len(premise_registry)} premises from corpus")


OUT_EDGES_JSONL = "tripartite_edges_all.jsonl"
OUT_THEOREMS_JSONL = "theorem_registry_all.jsonl"
OUT_PREMISES_JSONL = "premise_registry_unique.jsonl"

# Load all edges
with open(OUT_EDGES_JSONL, "r", encoding="utf-8") as f:
    edges = [json.loads(line) for line in f if line.strip()]

# Load all theorems
with open(OUT_THEOREMS_JSONL, "r", encoding="utf-8") as f:
    theorems = [json.loads(line) for line in f if line.strip()]

# Load all premises
with open(OUT_PREMISES_JSONL, "r", encoding="utf-8") as f:
    premises = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(edges)} edges")
print(f"Loaded {len(theorems)} theorems")
print(f"Loaded {len(premises)} premises")

In [24]:
"""Run categorized extraction on all proofs."""
import json
import sys
import importlib.util
from collections import Counter, defaultdict
from tqdm import tqdm

# Load data
print("Loading data...")
with open('complete_proofs.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f'Loaded {len(data)} proofs')

# Load myutils2
spec = importlib.util.spec_from_file_location('myutils2', '00_myutils2.py')
module = importlib.util.module_from_spec(spec)
sys.modules['myutils2'] = module
spec.loader.exec_module(module)

from myutils2 import extract_all_premises_categorized, extract_lemma_candidates_categorized

# Global accumulators
global_summary = {
    'global_lemmas': Counter(),
    'local_hypotheses': Counter(),
    'local_var_access': Counter(),
    'keywords_modifiers': Counter(),
    'type_class_annotations': Counter(),
    'corrupted_unicode': Counter(),
    'other': Counter(),
}

tactic_block_categorized = defaultdict(lambda: defaultdict(Counter))
tactic_blocks_in_proofs_cat = defaultdict(set)

# Process all proofs
print('Processing proofs...')
for i in tqdm(range(len(data)), desc='Categorized extraction'):
    result = extract_all_premises_categorized(i=i, data=data, printing=False)
    if result is None or result[0] is None:
        continue
    tactic_blocks_cat, proof_summary = result
    for cat_name, counter in proof_summary.items():
        global_summary[cat_name].update(counter)
    for tactic, categories in tactic_blocks_cat.items():
        tactic_blocks_in_proofs_cat[tactic].add(i)
        for cat_name, items in categories.items():
            for item in items:
                tactic_block_categorized[tactic][cat_name][item] += 1

# Build output
print("Building output...")
total_proofs = len(data)
categorized_output = {}
for tactic in tactic_block_categorized:
    num_proofs = len(tactic_blocks_in_proofs_cat[tactic])
    pct = (num_proofs / total_proofs) * 100 if total_proofs > 0 else 0
    categories_dict = {}
    category_totals = {}
    for cat_name in global_summary.keys():
        cat_data = dict(tactic_block_categorized[tactic][cat_name])
        categories_dict[cat_name] = cat_data
        category_totals[cat_name] = sum(cat_data.values())
    categorized_output[tactic] = {
        'tactic_block_count': num_proofs,
        'tactic_block_percentage_occurrence': pct,
        'categories': categories_dict,
        'category_totals': category_totals,
    }

# Sort by count
sorted_output = dict(sorted(categorized_output.items(), key=lambda x: x[1]['tactic_block_count'], reverse=True))

# Save
print("Saving categorized output...")
with open('00_full_tactic_to_premises_categorized.json', 'w', encoding='utf-8') as f:
    json.dump(sorted_output, f, indent=2, ensure_ascii=False)

print(f'Saved {len(sorted_output)} tactic blocks to 00_full_tactic_to_premises_categorized.json')

# Save global summary
global_summary_output = {
    cat_name: {
        'unique_count': len(counter),
        'total_occurrences': sum(counter.values()),
        'top_50': dict(counter.most_common(50)),
        'all_items': dict(counter)
    }
    for cat_name, counter in global_summary.items()
}
with open('00_global_category_summary.json', 'w', encoding='utf-8') as f:
    json.dump(global_summary_output, f, indent=2, ensure_ascii=False)

print('Saved global summary to 00_global_category_summary.json')

# Print stats
print()
print('=== GLOBAL SUMMARY ===')
for cat_name, counter in global_summary.items():
    unique = len(counter)
    total = sum(counter.values())
    print(f'{cat_name:25s}: {unique:6d} unique, {total:8d} total')

# Signal vs noise
print()
print('=== SIGNAL VS NOISE ===')
lemmas_unique = len(global_summary["global_lemmas"])
lemmas_total = sum(global_summary["global_lemmas"].values())
noise_cats = ["local_hypotheses", "local_var_access", "keywords_modifiers", "corrupted_unicode"]
noise_unique = sum(len(global_summary[c]) for c in noise_cats)
noise_total = sum(sum(global_summary[c].values()) for c in noise_cats)
print(f'Global lemmas (signal):   {lemmas_unique:6d} unique, {lemmas_total:8d} total')
print(f'Filtered items (noise):   {noise_unique:6d} unique, {noise_total:8d} total')
print(f'Signal/noise ratio: {lemmas_total/max(1,noise_total):.2f}x')

# Show top 10 tactic blocks
print()
print('=== TOP 10 TACTIC BLOCKS ===')
for i, (tactic, tdata) in enumerate(list(sorted_output.items())[:10]):
    print(f"\n{i+1}. Count: {tdata['tactic_block_count']} ({tdata['tactic_block_percentage_occurrence']:.3f}%)")
    print(f"   Tactic: {repr(tactic[:80])}")
    print(f"   Category totals: {tdata['category_totals']}")

print("\nDone!")

In [23]:
# Build a dictionary: {thm_name: [candidates]}, and also a total list of all candidates (all occurrences, no thm info)
from tqdm import tqdm
from collections import Counter, defaultdict

thm_to_candidates = {}
all_candidates_occurrences = []
# Track full tactic -> premises pairs across corpus
tactic_premises_pairs = []  # List of (tactic, premises_tuple) for counting
# Track which proofs contain each tactic block (for percentage calculation)
tactic_blocks_in_proofs = defaultdict(set)  # {tactic: set of proof indices}
# Track premise counts per tactic block
tactic_premise_counts = defaultdict(lambda: defaultdict(int))  # {tactic: {premise: count}}

# Add a progress bar for all data points
for i in tqdm(range(len(data)), desc="Extracting candidates", unit="proof"):
    result = extract_all_premises(
        i=i,
        data=data,
        printing=False
    )
    if result is None:
        # Stopped printing for each
        continue
    
    # Handle new return format: (all_candidates_list, tactic_to_premises)
    if isinstance(result, tuple):
        all_candidates, tactic_to_premises = result
    else:
        # Backward compatibility: if it's just a list
        all_candidates = result
        tactic_to_premises = {}

    # Theorem name assumed as data[i][0]
    thm = data[i][0]
    cands_in_proof = [candidate for _, candidate in all_candidates]
    thm_to_candidates[thm] = cands_in_proof

    # Add to the flat list of all candidate occurrences
    all_candidates_occurrences.extend(cands_in_proof)
    
    # Accumulate tactic -> premises pairs and track per proof
    for tactic, premises_list in tactic_to_premises.items():
        premises_tuple = tuple(premises_list)  # Convert to tuple for hashing
        tactic_premises_pairs.append((tactic, premises_tuple))
        
        # Track that this proof contains this tactic block
        tactic_blocks_in_proofs[tactic].add(i)
        
        # Count premises for this tactic block
        for premise in premises_list:
            tactic_premise_counts[tactic][premise] += 1

# Save the candidate occurrences counter to JSON, filtering to those found in corpus.jsonl
import json
from collections import Counter

# Load all lemma names from corpus.jsonl
allowed_lemmas_set = set()
with open("corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            file_entry = json.loads(line)
            premises = file_entry.get("premises", [])
            for prem in premises:
                full_name = prem.get("full_name", "")
                if full_name:
                    allowed_lemmas_set.add(full_name)
        except Exception:
            continue

occurrence_counter = Counter(all_candidates_occurrences)

# Filter keys to only those that are in the allowed lemmas set
filtered_occurrence_counter = {lemma: count for lemma, count in occurrence_counter.items() if lemma in allowed_lemmas_set}
# Also store the 'filtered out' (not in allowed_lemmas_set)
filtered_out_occurrence_counter = {lemma: count for lemma, count in occurrence_counter.items() if lemma not in allowed_lemmas_set}

# Save filtered counts
with open("lemma_candidate_counter.json", "w", encoding="utf-8") as f:
    json.dump(filtered_occurrence_counter, f, indent=2, ensure_ascii=False)

# Save filtered out counts
with open("lemma_candidate_counter_filtered_out.json", "w", encoding="utf-8") as f:
    json.dump(filtered_out_occurrence_counter, f, indent=2, ensure_ascii=False)

percent_filtered = (len(filtered_occurrence_counter) / max(1, len(occurrence_counter))) * 100
percent_filtered_out = (len(filtered_out_occurrence_counter) / max(1, len(occurrence_counter))) * 100
print(f"Filtered %: {percent_filtered:.2f}% ({len(filtered_occurrence_counter)} / {len(occurrence_counter)})")
print(f"Filtered out %: {percent_filtered_out:.2f}% ({len(filtered_out_occurrence_counter)} / {len(occurrence_counter)})")
print(f"Saved filtered candidate occurrences to lemma_candidate_counter.json")
print(f"Saved filtered out candidate occurrences to lemma_candidate_counter_filtered_out.json")

# Build nested dict structure: 
# {tactic_block: {
#    "tactic_block_count": int, 
#    "tactic_block_percentage_occurrence": float,
#    "extracted_premises": {premise: count},
#    "premise_percentages": {premise: %}}
# }
# Data is already tracked during the main loop above

# Total number of proofs
total_proofs = len(data)

# Build the nested dictionary structure
tactic_to_premises_data = {}
for tactic in tactic_blocks_in_proofs:
    # Count how many proofs contain this tactic block
    num_proofs_with_tactic = len(tactic_blocks_in_proofs[tactic])
    tactic_percentage = (num_proofs_with_tactic / total_proofs) * 100 if total_proofs > 0 else 0

    # Get all premises for this tactic and their counts
    premise_counts = dict(tactic_premise_counts[tactic])
    total_premise_occurrences = sum(premise_counts.values())

    # Calculate premise percentages (premise count / total premises for this tactic)
    premise_percentages = {}
    for premise, count in premise_counts.items():
        premise_percentage = (count / total_premise_occurrences) * 100 if total_premise_occurrences > 0 else 0
        premise_percentages[premise] = premise_percentage

    # Build the nested structure (put count first, then percentage as per prompt)
    tactic_to_premises_data[tactic] = {
        "tactic_block_count": num_proofs_with_tactic,
        "tactic_block_percentage_occurrence": tactic_percentage,
        "extracted_premises": premise_counts,
        "premise_percentages": premise_percentages
    }

# Save to JSON, sorted by tactic_block_count descending
sorted_tactic_to_premises = dict(
    sorted(
        tactic_to_premises_data.items(),
        key=lambda x: x[1]["tactic_block_count"],
        reverse=True
    )
)
with open("00_full_tactic_to_premises.json", "w", encoding="utf-8") as f:
    json.dump(sorted_tactic_to_premises, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(tactic_to_premises_data)} unique tactic blocks to 00_full_tactic_to_premises.json")
print(f"Total proofs processed: {total_proofs}")
print(f"Total tactic-premise pairs: {len(tactic_premises_pairs)}")

In [11]:
filtered_occurrence_counter

In [4]:
# # Compare v1 (raw) vs v2 (check full name first, then break down)
# from tqdm import tqdm
# import json
# from collections import Counter

# # Load data
# premise_registry = load_corpus_premises("corpus.jsonl")
# with open("complete_proofs.json", "r", encoding="utf-8") as f:
#     data = json.load(f)
# allowed_lemmas_set = {p.get("full_name", "") for p in premise_registry if p.get("full_name")}

# print(f"Loaded {len(premise_registry)} premises, {len(data)} proofs\n")

# # Extract candidates for both versions
# def extract_version(version, desc):
#     thm_to_cands, all_cands = {}, []
#     for i in tqdm(range(len(data)), desc=desc, unit="proof"):
#         cands = extract_all_premises(i=i, data=data, premise_registry=premise_registry if version=="v2" else None, 
#                                      printing=False, version=version)
#         if cands:
#             thm = data[i][0]
#             cand_list = [c for _, c in cands]
#             thm_to_cands[thm] = cand_list
#             all_cands.extend(cand_list)
#     return all_cands

# # Run both versions
# all_v1 = extract_version("v1", "V1: Raw extraction")
# all_v2 = extract_version("v2", "V2: Check corpus first")

# # Analyze and compare
# def analyze(candidates, name):
#     counter = Counter(candidates)
#     filtered = {k: v for k, v in counter.items() if k in allowed_lemmas_set}
#     filtered_out = {k: v for k, v in counter.items() if k not in allowed_lemmas_set}
#     pct = len(filtered)/max(1,len(counter))*100
#     print(f"{name}: {len(counter)} unique, {len(filtered)} in corpus ({pct:.1f}%)")
#     return filtered, filtered_out

# filtered_v1, out_v1 = analyze(all_v1, "V1")
# filtered_v2, out_v2 = analyze(all_v2, "V2")

# # Save results
# for name, filtered, filtered_out in [("v1", filtered_v1, out_v1), ("v2", filtered_v2, out_v2)]:
#     json.dump(filtered, open(f"lemma_candidate_counter_{name}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
#     json.dump(filtered_out, open(f"lemma_candidate_counter_filtered_out_{name}.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

# # Show differences
# v1_set, v2_set = set(Counter(all_v1).keys()), set(Counter(all_v2).keys())
# changed = [(c, c.rsplit(".", 1)[0]) for c in v1_set - v2_set if "." in c and c.rsplit(".", 1)[0] in v2_set][:10]
# if changed:
#     print("\nV2 broke down projections:")
#     for orig, proc in changed:
#         print(f"  {orig} -> {proc}")

In [ ]:
# # Save the candidate occurrences counter to JSON, filtering to those found in premise_registry_unique.jsonl
# import json
# from collections import Counter

# # Load all lemma names from premise_registry_unique.jsonl
# allowed_lemmas_set = set()
# with open("premise_registry_unique.jsonl", "r", encoding="utf-8") as f:
#     for line in f:
#         try:
#             entry = json.loads(line)
#             allowed_lemmas_set.add(entry["full_name"])
#         except Exception:
#             continue

# occurrence_counter = Counter(all_candidates_occurrences)

# # Filter keys to only those that are in the allowed lemmas set
# filtered_occurrence_counter = {lemma: count for lemma, count in occurrence_counter.items() if lemma in allowed_lemmas_set}
# # Also store the 'filtered out' (not in allowed_lemmas_set)
# filtered_out_occurrence_counter = {lemma: count for lemma, count in occurrence_counter.items() if lemma not in allowed_lemmas_set}

# # Save filtered counts
# with open("lemma_candidate_counter.json", "w", encoding="utf-8") as f:
#     json.dump(filtered_occurrence_counter, f, indent=2, ensure_ascii=False)

# # Save filtered out counts
# with open("lemma_candidate_counter_filtered_out.json", "w", encoding="utf-8") as f:
#     json.dump(filtered_out_occurrence_counter, f, indent=2, ensure_ascii=False)

# percent_filtered = (len(filtered_occurrence_counter) / max(1, len(occurrence_counter))) * 100
# percent_filtered_out = (len(filtered_out_occurrence_counter) / max(1, len(occurrence_counter))) * 100
# print(f"Filtered %: {percent_filtered:.2f}% ({len(filtered_occurrence_counter)} / {len(occurrence_counter)})")
# print(f"Filtered out %: {percent_filtered_out:.2f}% ({len(filtered_out_occurrence_counter)} / {len(occurrence_counter)})")
# print(f"Saved filtered candidate occurrences to lemma_candidate_counter.json")
# print(f"Saved filtered out candidate occurrences to lemma_candidate_counter_filtered_out.json")

In [ ]:
# =============================================================================
# CATEGORIZED PREMISE EXTRACTION
# =============================================================================
# This extracts all identifiers and categorizes them into:
# - global_lemmas: Qualified names with capital namespaces, known lemmas
# - local_hypotheses: Single letters, numbered (h1, h2), special (this, ih)
# - local_var_access: Patterns like x.property, this.method, f.rootSet
# - keywords_modifiers: Tactic names, only, using, etc.
# - type_class_annotations: Type names in type positions (MapsTo, Monoid, etc.)
# - corrupted_unicode: "â" and similar encoding garbage
# - other: Anything that doesn't fit elsewhere

# Force reload myutils2 to get the new categorized extraction functions
import sys
import importlib.util

if 'myutils2' in sys.modules:
    del sys.modules['myutils2']

spec = importlib.util.spec_from_file_location("myutils2", "00_myutils2.py")
module = importlib.util.module_from_spec(spec)
sys.modules["myutils2"] = module
spec.loader.exec_module(module)

from myutils2 import (
    extract_all_premises_categorized, 
    extract_lemma_candidates_categorized,
    split_tactic_blocks
)

print("Loaded categorized extraction functions")

In [ ]:
# Run categorized extraction on all proofs
from tqdm import tqdm
from collections import Counter, defaultdict
import json

# Data should already be loaded from earlier cells
print(f"Processing {len(data)} proofs with categorized extraction...")

# Global accumulators for each category
global_summary = {
    "global_lemmas": Counter(),
    "local_hypotheses": Counter(),
    "local_var_access": Counter(),
    "keywords_modifiers": Counter(),
    "type_class_annotations": Counter(),
    "corrupted_unicode": Counter(),
    "other": Counter(),
}

# Track per-tactic-block categorized data
# {tactic_block: {category: {item: count}}}
tactic_block_categorized = defaultdict(lambda: defaultdict(Counter))

# Track which proofs contain each tactic block
tactic_blocks_in_proofs_cat = defaultdict(set)

# Process all proofs
for i in tqdm(range(len(data)), desc="Categorized extraction", unit="proof"):
    result = extract_all_premises_categorized(i=i, data=data, printing=False)
    
    if result is None or result[0] is None:
        continue
    
    tactic_blocks_cat, proof_summary = result
    
    # Accumulate global summary
    for cat_name, counter in proof_summary.items():
        global_summary[cat_name].update(counter)
    
    # Accumulate per-tactic-block data
    for tactic, categories in tactic_blocks_cat.items():
        tactic_blocks_in_proofs_cat[tactic].add(i)
        for cat_name, items in categories.items():
            for item in items:
                tactic_block_categorized[tactic][cat_name][item] += 1

print(f"\nProcessed {len(data)} proofs")
print(f"Found {len(tactic_block_categorized)} unique tactic blocks")

# Print global summary
print("\n=== GLOBAL SUMMARY ===")
for cat_name, counter in global_summary.items():
    unique = len(counter)
    total = sum(counter.values())
    top_5 = counter.most_common(5)
    print(f"\n{cat_name}: {unique} unique, {total} total")
    print(f"  Top 5: {top_5}")

In [ ]:
# Build and save the categorized JSON structure
# Format:
# {
#   "tactic_block": {
#     "tactic_block_count": int,
#     "tactic_block_percentage_occurrence": float,
#     "categories": {
#       "global_lemmas": {"item": count, ...},
#       "local_hypotheses": {"item": count, ...},
#       "local_var_access": {"item": count, ...},
#       "keywords_modifiers": {"item": count, ...},
#       "type_class_annotations": {"item": count, ...},
#       "corrupted_unicode": {"item": count, ...},
#       "other": {"item": count, ...}
#     },
#     "category_totals": {
#       "global_lemmas": total_count,
#       "local_hypotheses": total_count,
#       ...
#     }
#   }
# }

total_proofs = len(data)

categorized_output = {}

for tactic in tactic_block_categorized:
    num_proofs_with_tactic = len(tactic_blocks_in_proofs_cat[tactic])
    tactic_percentage = (num_proofs_with_tactic / total_proofs) * 100 if total_proofs > 0 else 0
    
    # Build categories dict with actual counts
    categories_dict = {}
    category_totals = {}
    
    for cat_name in global_summary.keys():
        cat_data = dict(tactic_block_categorized[tactic][cat_name])
        categories_dict[cat_name] = cat_data
        category_totals[cat_name] = sum(cat_data.values())
    
    categorized_output[tactic] = {
        "tactic_block_count": num_proofs_with_tactic,
        "tactic_block_percentage_occurrence": tactic_percentage,
        "categories": categories_dict,
        "category_totals": category_totals,
    }

# Sort by tactic_block_count descending
sorted_output = dict(
    sorted(
        categorized_output.items(),
        key=lambda x: x[1]["tactic_block_count"],
        reverse=True
    )
)

# Save to JSON
output_file = "00_full_tactic_to_premises_categorized.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(sorted_output, f, indent=2, ensure_ascii=False)

print(f"Saved categorized extraction to {output_file}")
print(f"Total unique tactic blocks: {len(sorted_output)}")

# Show top 10 tactics
print("\n=== TOP 10 TACTIC BLOCKS ===")
for i, (tactic, data) in enumerate(list(sorted_output.items())[:10]):
    print(f"\n{i+1}. Count: {data['tactic_block_count']} ({data['tactic_block_percentage_occurrence']:.3f}%)")
    print(f"   Tactic: {repr(tactic[:80])}")
    print(f"   Category totals: {data['category_totals']}")

In [ ]:
# Save global summary separately
global_summary_output = {
    cat_name: {
        "unique_count": len(counter),
        "total_occurrences": sum(counter.values()),
        "top_50": dict(counter.most_common(50)),
        "all_items": dict(counter)
    }
    for cat_name, counter in global_summary.items()
}

with open("00_global_category_summary.json", "w", encoding="utf-8") as f:
    json.dump(global_summary_output, f, indent=2, ensure_ascii=False)

print("Saved global category summary to 00_global_category_summary.json")

# Calculate overall statistics
total_unique = sum(len(c) for c in global_summary.values())
total_occurrences = sum(sum(c.values()) for c in global_summary.values())

print(f"\n=== OVERALL STATISTICS ===")
print(f"Total unique identifiers across all categories: {total_unique}")
print(f"Total identifier occurrences: {total_occurrences}")

print("\n=== CATEGORY BREAKDOWN ===")
for cat_name, counter in global_summary.items():
    unique = len(counter)
    total = sum(counter.values())
    pct_unique = (unique / total_unique * 100) if total_unique > 0 else 0
    pct_total = (total / total_occurrences * 100) if total_occurrences > 0 else 0
    print(f"{cat_name:25s}: {unique:6d} unique ({pct_unique:5.1f}%), {total:8d} total ({pct_total:5.1f}%)")

# Show breakdown of what's being extracted vs filtered
lemmas_unique = len(global_summary["global_lemmas"])
lemmas_total = sum(global_summary["global_lemmas"].values())
noise_unique = sum(len(global_summary[c]) for c in ["local_hypotheses", "local_var_access", "keywords_modifiers", "corrupted_unicode"])
noise_total = sum(sum(global_summary[c].values()) for c in ["local_hypotheses", "local_var_access", "keywords_modifiers", "corrupted_unicode"])

print(f"\n=== SIGNAL VS NOISE ===")
print(f"Global lemmas (signal):   {lemmas_unique:6d} unique, {lemmas_total:8d} total")
print(f"Filtered items (noise):   {noise_unique:6d} unique, {noise_total:8d} total")
print(f"Signal/noise ratio: {lemmas_total/max(1,noise_total):.2f}x")

In [ ]:
# from collections import Counter

# # Print the top 20 most common entries in filtered_out_occurrence_counter
# top_20 = Counter(filtered_out_occurrence_counter).most_common(20)
# for lemma, count in top_20:
#     print(f"{lemma}: {count}")

## using corpus we already find more matched lemmas

In [ ]:
from collections import Counter

# Print the top 20 most common entries in filtered_out_occurrence_counter
top_20 = Counter(filtered_out_occurrence_counter).most_common(20)
for lemma, count in top_20:
    print(f"{lemma}: {count}")

## running proof with tactics and resolving at proof time? why are we doing this

In [7]:
theorem_registry=theorems
# Run proof tactics with the new signature
result = run_proof_tactics(
    i=0,  # proof index
    data=data,
    theorem_registry=theorem_registry,  # Use theorem_registry instead of edges
    premise_registry=premise_registry,  # Use premise_registry instead of edges
    printing=True,
    print_states=False,
    show_new_lemmas_per_step=True
)

# Access results
print(f"Success: {result['success']}")
print(f"Theorem: {result['theorem_full_name']}")
print(f"Resolved lemmas: {len(result['resolved_best'])}")
print(f"Unresolved: {len(result['unresolved'])}")

In [ ]:

# Run proof tactics for a specific proof
result = run_proof_tactics(
    i=0,  # proof index
    data=data,
    edges=edges,
    printing=True,
    print_states=False,
    show_new_lemmas_per_step=True
)

# Access results
print(f"Success: {result['success']}")
print(f"Theorem: {result['theorem_full_name']}")
print(f"Resolved lemmas: {len(result['resolved_best'])}")
print(f"Unresolved: {len(result['unresolved'])}")